# 00 — Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

In [ ]:
PROJECT_ROOT = Path("../").resolve()

config_path = PROJECT_ROOT / "config" / "idioms.yaml"
with open(config_path, "r") as f:
    idiom_config = yaml.safe_load(f)

idioms = idiom_config.get("idioms", [])
print(f"Idiom count: {len(idioms)}")

In [ ]:
candidates_path = PROJECT_ROOT / "data" / "interim" / "candidates.parquet"
candidates = pd.read_parquet(candidates_path)

print("Shape:", candidates.shape)
print("\nDtypes:")
print(candidates.dtypes)
candidates.head(5)

In [ ]:
idiom_counts = candidates["idiom"].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
idiom_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Candidate Counts per Idiom", fontsize=14)
ax.set_xlabel("Idiom")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
year_counts = candidates.groupby("year").size()

fig, ax = plt.subplots(figsize=(12, 4))
year_counts.plot(kind="line", ax=ax, color="steelblue", linewidth=1.5)
ax.set_title("Candidate Count by Year", fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Count")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
observations_path = PROJECT_ROOT / "data" / "interim" / "observations.parquet"

if observations_path.exists():
    observations = pd.read_parquet(observations_path)
    print("is_idiomatic value counts:")
    print(observations["is_idiomatic"].value_counts())
    print("\nConfidence distribution:")
    print(observations["confidence"].describe())

    fig, ax = plt.subplots(figsize=(8, 4))
    observations["confidence"].hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title("Confidence Score Distribution", fontsize=14)
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.show()
else:
    print("observations.parquet not found — run stage 02 first.")

In [ ]:
if observations_path.exists():
    crosstab = pd.crosstab(observations["idiom"], observations["is_idiomatic"])
    display(crosstab)
else:
    print("observations.parquet not found — skipping cross-tab.")

In [ ]:
scores_path = PROJECT_ROOT / "data" / "processed" / "scores.parquet"
scores = pd.read_parquet(scores_path)

fig, ax = plt.subplots(figsize=(12, 5))
groups = scores.groupby(["model", "denomination"])
labels = []
data = []
for (model, denom), grp in groups:
    labels.append(f"{model}\n{denom}")
    data.append(grp["score_axis"].dropna().values)

ax.boxplot(data, labels=labels, patch_artist=True)
ax.set_title("score_axis by Model and Denomination", fontsize=14)
ax.set_xlabel("Model / Denomination")
ax.set_ylabel("score_axis")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Observations

Add narrative here.